# HybridOnly vs HybridGraph + Generate Evaluation

This notebook runs the QA benchmark and compares two modes:

- `HybridOnly`: dense FAISS + BM25 10 shards, fused with RRF, then answer generation.
- `HybridGraph`: HybridOnly seeds, GraphExpansion context expansion, then answer generation.

Main metrics:
- Retrieval: `recall`, `hit`, `mrr`, `ndcg`, `precision` at `k = 1, 3, 5, 10`.
- Generation: `exact_match`, `token_f1`, `rouge_l`.
- Answerability: answerable/unanswerable split and `unanswerable_accuracy`.
- Latency: embedding, dense search, BM25 search, fusion, graph expansion, generation, total.


In [ ]:
!pip -q install faiss-cpu sentence-transformers pandas psutil rank-bm25 underthesea openai


In [ ]:
from pathlib import Path
import gc
import contextlib
import io
import json
import math
import os
import pickle
import re
import shutil
import sys
import time
import unicodedata
from collections import Counter
from dataclasses import dataclass

import numpy as np
import pandas as pd
import psutil
import torch
from kaggle_secrets import UserSecretsClient
from openai import OpenAI
from sentence_transformers import SentenceTransformer


In [ ]:
def print_ram(label):
    mem = psutil.virtual_memory()
    process_ram = psutil.Process(os.getpid()).memory_info().rss / 1024**3
    print(f'[{label}] process={process_ram:.2f}GB | free={mem.available / 1024**3:.2f}GB')

print_ram('Start')


In [ ]:
REPO_DIR = Path('/kaggle/working/TextMining')

if not REPO_DIR.exists():
    !git clone -q https://github.com/PhuongThao-2005/TextMining.git /kaggle/working/TextMining

sys.path.insert(0, str(REPO_DIR / 'src'))

from knowledge_graph import GraphExpansion, load_knowledge_graph
from retrieval.sqlite_faiss_store import SQLitePayloadFaissVectorStore

print('Repo ready:', REPO_DIR)


In [ ]:
KAGGLE_INPUT = Path('/kaggle/input')


def first_existing_path(candidates):
    for path in candidates:
        path = Path(path)
        if path.exists():
            return path
    raise FileNotFoundError('None of these paths exist:\n' + '\n'.join(str(p) for p in candidates))


def find_input_dir_with_files(slug_candidates, required_relative_files):
    candidates = []
    for slug in slug_candidates:
        candidates.append(KAGGLE_INPUT / slug)
        candidates.append(KAGGLE_INPUT / 'datasets' / slug)
    for base in candidates:
        if base.exists() and all((base / rel).exists() for rel in required_relative_files):
            return base

    # Fallback: scan mounted Kaggle inputs one level deep. This handles Kaggle renaming folders.
    for base in sorted([p for p in KAGGLE_INPUT.glob('*') if p.is_dir()]):
        if all((base / rel).exists() for rel in required_relative_files):
            return base
    raise FileNotFoundError(
        'Could not find Kaggle input folder containing: ' + ', '.join(str(x) for x in required_relative_files)
    )


def find_input_file(slug_candidates, filename):
    candidates = []
    for slug in slug_candidates:
        candidates.append(KAGGLE_INPUT / slug / filename)
        candidates.append(KAGGLE_INPUT / 'datasets' / slug / filename)
    for path in candidates:
        if path.exists():
            return path
    matches = sorted(KAGGLE_INPUT.glob(f'*/{filename}'))
    if matches:
        return matches[0]
    raise FileNotFoundError(f'Could not find {filename} under /kaggle/input')


FAISS_INPUT_DIR = find_input_dir_with_files(
    ['faiss-chunk-meta', 'kittrntunk/faiss-chunk-meta', 'faisse-only-chunk', 'kittrntunk/faisse-only-chunk'],
    ['index.faiss', 'payloads.jsonl', 'id_map.json'],
)
FAISS_WORK_DIR = Path('/kaggle/working/faiss-chunk-meta')
BM25_INPUT_DIR = find_input_dir_with_files(
    ['bm25-tokenized', 'nguyenlethienlyy/bm25-tokenized/bm25'],
    ['shard_00/bm25_index.pkl', 'shard_00/bm25_metadata.pkl'],
)
GRAPH_PATH = find_input_file(
    ['legalrag-knowledge-graph', 'nguyenlethienlyy/legalrag-knowledge-graph', 'faiss-retrieve-generate-graph', 'nguyenlethienlyy/faiss-retrieve-generate-graph'],
    'knowledge_graph.gpickle',
)
QA_PATH = find_input_file(
    ['legalrag-text-mining-benchmark', 'nguyenlethienlyy/legalrag-text-mining-benchmark'],
    'qa_final.jsonl',
)

OUT_DIR = Path('/kaggle/working/evaluation_runs/ablation/hybrid_graph_generate_eval')

EMBEDDING_MODEL = 'intfloat/multilingual-e5-large'
GENERATION_MODEL = 'gpt-4o-mini'
BASE_URL = 'https://api.shopaikey.com/v1'

SEARCH_K = 50
FINAL_TOP_N = 10
K_VALUES = [1, 3, 5, 10]
RRF_K = 60
GRAPH_SEED_N = 5
GRAPH_MAX_HOP = 1
GRAPH_CONTEXT = 10
FILTER_PROFILE = 'broad'
SAMPLE_LIMIT = None  # change to 10 for a quick smoke test; None runs the full benchmark
EVAL_START = 0      # inclusive; use 0, 100, 200... to split runs
EVAL_END = None     # exclusive; use 100, 200, 300...; None means until the end
BM25_PRECOMPUTE = True  # load each BM25 shard once and search every question in this run

# BM25 is large. Keep this at 1 on Kaggle unless you know the runtime has enough RAM.
BM25_MAX_SHARDS_IN_MEMORY = 1

bm25_shards = sorted([p for p in BM25_INPUT_DIR.glob('shard_*') if p.is_dir()])
if len(bm25_shards) != 10:
    raise FileNotFoundError(f'Expected 10 BM25 shards under {BM25_INPUT_DIR}, found {len(bm25_shards)}')
for shard in bm25_shards:
    for name in ['bm25_index.pkl', 'bm25_metadata.pkl']:
        if not (shard / name).exists():
            raise FileNotFoundError(shard / name)

OUT_DIR.mkdir(parents=True, exist_ok=True)

print('FAISS_INPUT_DIR =', FAISS_INPUT_DIR)
print('BM25_INPUT_DIR  =', BM25_INPUT_DIR)
print('GRAPH_PATH      =', GRAPH_PATH)
print('QA_PATH         =', QA_PATH)
print('OUT_DIR         =', OUT_DIR)


In [ ]:
FAISS_WORK_DIR.mkdir(parents=True, exist_ok=True)

for name in ['index.faiss', 'payloads.jsonl', 'id_map.json']:
    src = FAISS_INPUT_DIR / name
    dst = FAISS_WORK_DIR / name
    if not dst.exists():
        os.symlink(src, dst)

cache_src = FAISS_INPUT_DIR / 'payload_cache.sqlite'
cache_dst = FAISS_WORK_DIR / 'payload_cache.sqlite'
if not cache_dst.exists():
    shutil.copy2(cache_src, cache_dst)

print('FAISS working folder ready:', FAISS_WORK_DIR)
print_ram('After preparing FAISS folder')


In [ ]:
all_qa_rows = []
with QA_PATH.open('r', encoding='utf-8') as f:
    for line in f:
        if line.strip():
            all_qa_rows.append(json.loads(line))

qa_total_available = len(all_qa_rows)

if SAMPLE_LIMIT is not None:
    all_qa_rows = all_qa_rows[:SAMPLE_LIMIT]

run_start_index = max(0, int(EVAL_START or 0))
run_end_index = len(all_qa_rows) if EVAL_END is None else min(len(all_qa_rows), int(EVAL_END))
qa_rows = all_qa_rows[run_start_index:run_end_index]

RUN_LABEL = f'q{run_start_index:04d}_q{run_end_index:04d}'
OUT_DIR = OUT_DIR / RUN_LABEL
OUT_DIR.mkdir(parents=True, exist_ok=True)

answerable_count = sum(bool((row.get('ground_truth') or {}).get('chunk_ids')) for row in qa_rows)

print('QA available   =', qa_total_available)
print('QA selected    =', len(qa_rows), '| range:', run_start_index, '->', run_end_index)
print('answerable     =', answerable_count)
print('unanswerable   =', len(qa_rows) - answerable_count)
print('OUT_DIR        =', OUT_DIR)


In [ ]:
t0 = time.perf_counter()
store = SQLitePayloadFaissVectorStore.load(FAISS_WORK_DIR)
print(f'Loaded FAISS store: {store.total_vectors:,} vectors, {time.perf_counter() - t0:.1f}s')
print_ram('After FAISS store')


In [ ]:
t0 = time.perf_counter()
graph = load_knowledge_graph(GRAPH_PATH).graph
graph_expansion = GraphExpansion(graph)
print(f'Loaded graph: {len(graph.chunks):,} chunks, {time.perf_counter() - t0:.1f}s')
print_ram('After graph')


In [ ]:
def simple_tokenize(text):
    text = unicodedata.normalize('NFC', str(text or '')).lower()
    text = re.sub(r'[^\w\s]', ' ', text, flags=re.UNICODE)
    return [tok for tok in text.split() if len(tok) > 1]


def get_tokenizer():
    try:
        from underthesea import word_tokenize

        def tokenize(text):
            return simple_tokenize(word_tokenize(str(text or ''), format='text'))

        print('Using underthesea tokenizer for BM25 queries')
        return tokenize
    except Exception:
        print('Using simple tokenizer for BM25 queries')
        return simple_tokenize


@dataclass
class SearchHit:
    point_id: str
    score: float
    payload: dict


class BM25Shard:
    def __init__(self, shard_dir):
        self.shard_dir = Path(shard_dir)
        with (self.shard_dir / 'bm25_index.pkl').open('rb') as f:
            self.bm25 = pickle.load(f)
        with (self.shard_dir / 'bm25_metadata.pkl').open('rb') as f:
            meta = pickle.load(f)
        self.chunk_ids = meta['chunk_ids']
        self.payloads = meta['payloads']

    def search_tokens(self, query_tokens, top_k):
        scores = self.bm25.get_scores(query_tokens)
        if len(scores) == 0:
            return []

        n = min(top_k, len(scores))
        candidate_indices = np.argpartition(scores, -n)[-n:]
        candidate_indices = candidate_indices[np.argsort(scores[candidate_indices])[::-1]]

        hits = []
        for idx in candidate_indices:
            idx = int(idx)
            score = float(scores[idx])
            if score <= 0:
                continue
            hits.append(SearchHit(str(self.chunk_ids[idx]), score, self.payloads[idx]))
        del scores
        return hits


class BatchShardedBM25Retriever:
    """Memory-safe BM25: load each shard once, search all selected questions, then unload."""

    def __init__(self, shard_dirs):
        self.shard_dirs = list(shard_dirs)
        self.tokenizer = get_tokenizer()
        self.cache = {}
        print(f'BM25 batch mode: {len(self.shard_dirs)} shards; one shard in RAM at a time')

    @staticmethod
    def keep_top_hits(candidates, top_k):
        best = {}
        for hit in candidates:
            chunk_id = str(hit.payload.get('chunk_id') or hit.point_id)
            if chunk_id not in best or hit.score > best[chunk_id].score:
                best[chunk_id] = hit
        return sorted(best.values(), key=lambda h: h.score, reverse=True)[:top_k]

    def precompute(self, questions, top_k=50):
        t0 = time.perf_counter()
        unique_questions = list(dict.fromkeys(str(q or '') for q in questions))
        query_tokens = {q: self.tokenizer(q) for q in unique_questions}
        candidates_by_query = {q: [] for q in unique_questions}

        for shard_no, shard_dir in enumerate(self.shard_dirs, start=1):
            gc.collect()
            shard_t0 = time.perf_counter()
            shard = BM25Shard(shard_dir)
            doc_count = len(shard.chunk_ids)
            try:
                for q in unique_questions:
                    candidates_by_query[q].extend(shard.search_tokens(query_tokens[q], top_k=top_k))
            finally:
                del shard
                gc.collect()
            print(f'BM25 shard {shard_no:02d}/{len(self.shard_dirs):02d} ({Path(shard_dir).name}): searched {len(unique_questions):,} questions over {doc_count:,} docs in {time.perf_counter() - shard_t0:.1f}s')

        self.cache = {
            q: self.keep_top_hits(candidates, top_k)
            for q, candidates in candidates_by_query.items()
        }
        del candidates_by_query
        gc.collect()
        return time.perf_counter() - t0

    def search_with_latency(self, query, top_k=50):
        query = str(query or '')
        if query in self.cache:
            return self.cache[query][:top_k], 0.0

        # Fallback for ad-hoc questions after precompute.
        t0 = time.perf_counter()
        query_tokens = self.tokenizer(query)
        candidates = []
        for shard_dir in self.shard_dirs:
            shard = BM25Shard(shard_dir)
            try:
                candidates.extend(shard.search_tokens(query_tokens, top_k=top_k))
            finally:
                del shard
                gc.collect()
        hits = self.keep_top_hits(candidates, top_k)
        del candidates
        gc.collect()
        return hits, time.perf_counter() - t0


t0 = time.perf_counter()
bm25 = BatchShardedBM25Retriever(bm25_shards)
BM25_PRECOMPUTE_LATENCY_SEC = 0.0
BM25_AMORTIZED_LATENCY_SEC = 0.0

if BM25_PRECOMPUTE:
    questions_for_bm25 = [row.get('question') or '' for row in qa_rows]
    BM25_PRECOMPUTE_LATENCY_SEC = bm25.precompute(questions_for_bm25, top_k=SEARCH_K)
    BM25_AMORTIZED_LATENCY_SEC = BM25_PRECOMPUTE_LATENCY_SEC / max(1, len(qa_rows))
    print(f'BM25 precomputed for {len(questions_for_bm25):,} questions in {BM25_PRECOMPUTE_LATENCY_SEC/60:.1f} min')
    print(f'BM25 amortized latency: {BM25_AMORTIZED_LATENCY_SEC:.3f}s/query')

print(f'BM25 ready in {time.perf_counter() - t0:.1f}s')
print_ram('After BM25 batch precompute')


In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

t0 = time.perf_counter()
embedder = SentenceTransformer(EMBEDDING_MODEL, device=device)
print(f'Embedder ready: {EMBEDDING_MODEL} | device={device} | {time.perf_counter() - t0:.1f}s')
print_ram('After embedder')


In [ ]:
api_key = UserSecretsClient().get_secret('OPENAI_API_KEY').strip()
client = OpenAI(api_key=api_key, base_url=BASE_URL)

print('Generator ready:', GENERATION_MODEL)
print('BASE_URL:', BASE_URL)


In [ ]:
def filters_for(profile):
    if profile == 'current_law':
        return {'validity_group': {'in': ['active', 'partial', 'future']}}
    if profile == 'historical':
        return {'validity_group': {'in': ['expired', 'active', 'partial']}}
    return {'validity_group': {'in': ['active', 'partial', 'future', 'expired', 'unknown']}}


def row_from_payload(payload, rank, score=None, source=''):
    return {
        'rank': rank,
        'score': None if score is None else float(score),
        'source': source,
        'chunk_id': str(payload.get('chunk_id') or ''),
        'document_id': str(payload.get('id_str') or ''),
        'provision_id': str(payload.get('parent_unit_id') or ''),
        'citation': payload.get('citation_anchor') or payload.get('citation_label') or '',
        'title': payload.get('title') or '',
        'validity': payload.get('validity_group') or '',
        'text': payload.get('chunk_text') or '',
    }


def retrieve_dense_hits(question):
    t0 = time.perf_counter()
    vector = embedder.encode(['query: ' + question], normalize_embeddings=True).astype('float32')[0].tolist()
    embed_latency = time.perf_counter() - t0

    t0 = time.perf_counter()
    with contextlib.redirect_stdout(io.StringIO()):
        hits = store.search(vector, limit=SEARCH_K, score_threshold=0.0, filters=filters_for(FILTER_PROFILE))
    dense_latency = time.perf_counter() - t0
    return hits, embed_latency, dense_latency


def rrf_fuse(dense_hits, sparse_hits, top_n=10, rrf_k=60):
    scores = {}
    best_hit = {}
    sources = {}
    for rank, hit in enumerate(dense_hits, start=1):
        chunk_id = str(hit.payload.get('chunk_id') or hit.point_id)
        scores[chunk_id] = scores.get(chunk_id, 0.0) + 1.0 / (rrf_k + rank)
        best_hit.setdefault(chunk_id, hit)
        sources.setdefault(chunk_id, set()).add('dense')
    for rank, hit in enumerate(sparse_hits, start=1):
        chunk_id = str(hit.payload.get('chunk_id') or hit.point_id)
        scores[chunk_id] = scores.get(chunk_id, 0.0) + 1.0 / (rrf_k + rank)
        best_hit.setdefault(chunk_id, hit)
        sources.setdefault(chunk_id, set()).add('bm25')
    ordered_ids = sorted(scores, key=scores.get, reverse=True)[:top_n]
    rows = []
    for chunk_id in ordered_ids:
        hit = best_hit[chunk_id]
        rows.append(row_from_payload(hit.payload, len(rows) + 1, scores[chunk_id], '+'.join(sorted(sources[chunk_id]))))
    return rows


def retrieve_hybrid(question):
    dense_hits, embed_latency, dense_latency = retrieve_dense_hits(question)
    sparse_hits, sparse_latency = bm25.search_with_latency(question, top_k=SEARCH_K)
    if BM25_PRECOMPUTE and sparse_latency == 0.0:
        sparse_latency = BM25_AMORTIZED_LATENCY_SEC
    t0 = time.perf_counter()
    rows = rrf_fuse(dense_hits, sparse_hits, top_n=FINAL_TOP_N, rrf_k=RRF_K)
    fusion_latency = time.perf_counter() - t0
    return rows, embed_latency, dense_latency, sparse_latency, fusion_latency


In [ ]:
def expand_with_graph(seed_rows):
    t0 = time.perf_counter()
    seed_ids = [row['chunk_id'] for row in seed_rows]
    expanded = graph_expansion.expand(seed_ids, max_hop=GRAPH_MAX_HOP, max_context=GRAPH_CONTEXT)
    chunk_ids = list(expanded.ordered_context_chunks)

    if not chunk_ids:
        return seed_rows, time.perf_counter() - t0, 0, list(expanded.warnings)

    with contextlib.redirect_stdout(io.StringIO()):
        hits = store.scroll({'chunk_id': {'in': chunk_ids}}, limit=len(chunk_ids))

    payload_by_chunk_id = {str(hit.payload.get('chunk_id')): hit.payload for hit in hits}
    rows = []
    for chunk_id in chunk_ids:
        payload = payload_by_chunk_id.get(chunk_id)
        if payload:
            rows.append(row_from_payload(payload, len(rows) + 1))

    seed_set = set(seed_ids)
    graph_added = len([row for row in rows if row['chunk_id'] not in seed_set])
    return rows or seed_rows, time.perf_counter() - t0, graph_added, list(expanded.warnings)


In [ ]:
def format_context(rows):
    blocks = []
    for row in rows:
        blocks.append(
            f"[{row['rank']}] {row['citation']} - {row['title']}\n"
            f"chunk_id={row['chunk_id']}\n"
            f"{row['text']}"
        )
    return '\n\n'.join(blocks)

def generate_answer(question, context_rows):
    prompt = f"""Bạn là trợ lý pháp lý tiếng Việt trong hệ thống RAG.

Hãy trả lời QUESTION dựa trên CONTEXT được cung cấp.

Nguyên tắc:
- Chỉ dùng thông tin có trong CONTEXT, không dùng kiến thức bên ngoài.
- Không bịa thêm căn cứ, điều kiện, ngoại lệ hoặc số điều nếu CONTEXT không nêu.
- Nếu CONTEXT không đủ thông tin để trả lời, hãy nói: "Không có đủ thông tin trong ngữ cảnh được cung cấp."
- Nếu CONTEXT chỉ trả lời được một phần câu hỏi, hãy nói rõ phạm vi đó.

Cách trả lời:
- Trả lời tự nhiên, rõ ràng, bằng tiếng Việt có dấu.
- Ưu tiên trả lời trực tiếp trước, giải thích ngắn sau nếu cần.
- Khi có căn cứ pháp lý trong CONTEXT, hãy nêu căn cứ ở cuối câu trả lời.
- Không trình bày quá trình suy luận nội bộ.

QUESTION:
{question}

CONTEXT:
{format_context(context_rows)}

Trả lời:""".strip()

    t0 = time.perf_counter()
    response = client.chat.completions.create(
        model=GENERATION_MODEL,
        messages=[{'role': 'user', 'content': prompt}],
        temperature=0,
    )
    latency = time.perf_counter() - t0
    return response.choices[0].message.content.strip(), latency


In [ ]:
def normalize_text(text):
    text = (text or '').lower()
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def token_f1(prediction, reference):
    pred_tokens = normalize_text(prediction).split()
    ref_tokens = normalize_text(reference).split()
    if not pred_tokens or not ref_tokens:
        return float(pred_tokens == ref_tokens)
    common = {}
    for token in pred_tokens:
        common[token] = min(pred_tokens.count(token), ref_tokens.count(token))
    overlap = sum(common.values())
    if overlap == 0:
        return 0.0
    precision = overlap / len(pred_tokens)
    recall = overlap / len(ref_tokens)
    return 2 * precision * recall / (precision + recall)

def rouge_l(prediction, reference):
    a = normalize_text(prediction).split()
    b = normalize_text(reference).split()
    if not a or not b:
        return float(a == b)
    dp = [[0] * (len(b) + 1) for _ in range(len(a) + 1)]
    for i in range(1, len(a) + 1):
        for j in range(1, len(b) + 1):
            dp[i][j] = dp[i-1][j-1] + 1 if a[i-1] == b[j-1] else max(dp[i-1][j], dp[i][j-1])
    lcs = dp[-1][-1]
    precision = lcs / len(a)
    recall = lcs / len(b)
    return 2 * precision * recall / (precision + recall) if precision + recall else 0.0

def exact_match(prediction, reference):
    return float(normalize_text(prediction) == normalize_text(reference))

def has_fallback_answer(answer):
    return 'không có đủ thông tin' in normalize_text(answer)


In [ ]:
def recall_at_k(retrieved, relevant, k):
    relevant = set(relevant)
    return len(set(retrieved[:k]) & relevant) / len(relevant) if relevant else 0.0


def hit_at_k(retrieved, relevant, k):
    return 1.0 if relevant and (set(retrieved[:k]) & set(relevant)) else 0.0


def mrr_at_k(retrieved, relevant, k):
    relevant = set(relevant)
    if not relevant:
        return 0.0
    for rank, chunk_id in enumerate(retrieved[:k], start=1):
        if chunk_id in relevant:
            return 1.0 / rank
    return 0.0


def ndcg_at_k(retrieved, relevant, k):
    relevant = set(relevant)
    if not relevant:
        return 0.0
    dcg = 0.0
    for rank, chunk_id in enumerate(retrieved[:k], start=1):
        if chunk_id in relevant:
            dcg += 1.0 / math.log2(rank + 1)
    ideal_hits = min(len(relevant), k)
    ideal = sum(1.0 / math.log2(rank + 1) for rank in range(1, ideal_hits + 1))
    return dcg / ideal if ideal else 0.0


def precision_at_k(retrieved, relevant, k):
    return len(set(retrieved[:k]) & set(relevant)) / k if k else 0.0


def add_retrieval_metrics(row, retrieved, relevant):
    for k in K_VALUES:
        row[f'recall@{k}'] = recall_at_k(retrieved, relevant, k)
        row[f'hit@{k}'] = hit_at_k(retrieved, relevant, k)
        row[f'mrr@{k}'] = mrr_at_k(retrieved, relevant, k)
        row[f'ndcg@{k}'] = ndcg_at_k(retrieved, relevant, k)
        row[f'precision@{k}'] = precision_at_k(retrieved, relevant, k)
    return row


def build_output_row(qa, mode, context_rows, answer, timings, graph_info):
    gt = qa.get('ground_truth') or {}
    relevant = [str(x) for x in (gt.get('chunk_ids') or []) if x]
    retrieved = [row['chunk_id'] for row in context_rows]
    reference = qa.get('reference_answer') or ''
    answerable = bool(relevant)
    row = {
        'qa_id': qa.get('qa_id'),
        'mode': mode,
        'question': qa.get('question'),
        'reference_answer': reference,
        'generated_answer': answer,
        'answer_type': qa.get('answer_type'),
        'category': qa.get('category'),
        'difficulty': qa.get('difficulty'),
        'is_answerable': answerable,
        'ground_truth_chunk_ids': relevant,
        'top_k_chunks': context_rows,
        'retrieved_chunk_ids': retrieved,
        'retrieved_count': len(retrieved),
        'exact_match': exact_match(answer, reference),
        'token_f1': token_f1(answer, reference),
        'rouge_l': rouge_l(answer, reference),
        'unanswerable_correct': (not answerable and has_fallback_answer(answer)),
        **timings,
        **graph_info,
    }
    return add_retrieval_metrics(row, retrieved, relevant)


In [ ]:
all_cases = []
run_start = time.perf_counter()

for index, qa in enumerate(qa_rows, start=1):
    question = qa.get('question') or ''
    hybrid_rows, embed_latency, dense_latency, bm25_latency, fusion_latency = retrieve_hybrid(question)

    hybrid_answer, hybrid_gen_latency = generate_answer(question, hybrid_rows)
    all_cases.append(build_output_row(
        qa,
        'HybridOnly',
        hybrid_rows,
        hybrid_answer,
        {
            'embedding_latency_sec': embed_latency,
            'dense_search_latency_sec': dense_latency,
            'bm25_search_latency_sec': bm25_latency,
            'fusion_latency_sec': fusion_latency,
            'graph_latency_sec': 0.0,
            'generation_latency_sec': hybrid_gen_latency,
            'total_latency_sec': embed_latency + dense_latency + bm25_latency + fusion_latency + hybrid_gen_latency,
        },
        {
            'seed_count': 0,
            'graph_added_count': 0,
            'graph_warning_count': 0,
            'graph_warnings': [],
        },
    ))

    graph_rows, graph_latency, graph_added_count, warnings = expand_with_graph(hybrid_rows[:GRAPH_SEED_N])
    graph_answer, graph_gen_latency = generate_answer(question, graph_rows)
    all_cases.append(build_output_row(
        qa,
        'HybridGraph',
        graph_rows,
        graph_answer,
        {
            'embedding_latency_sec': embed_latency,
            'dense_search_latency_sec': dense_latency,
            'bm25_search_latency_sec': bm25_latency,
            'fusion_latency_sec': fusion_latency,
            'graph_latency_sec': graph_latency,
            'generation_latency_sec': graph_gen_latency,
            'total_latency_sec': embed_latency + dense_latency + bm25_latency + fusion_latency + graph_latency + graph_gen_latency,
        },
        {
            'seed_count': min(len(hybrid_rows), GRAPH_SEED_N),
            'graph_added_count': graph_added_count,
            'graph_warning_count': len(warnings),
            'graph_warnings': warnings,
        },
    ))

    if index % 10 == 0 or index == len(qa_rows):
        elapsed = time.perf_counter() - run_start
        print(f'{index}/{len(qa_rows)} done | elapsed={elapsed/60:.1f} min')
        print_ram(f'After {index} questions')

total_benchmark_time = time.perf_counter() - run_start
print(f'Done in {total_benchmark_time/60:.1f} minutes')


In [ ]:
cases_df = pd.DataFrame(all_cases)


def average(rows, key):
    values = [row.get(key) for row in rows if isinstance(row.get(key), (int, float))]
    return sum(values) / len(values) if values else 0.0


def percentile(rows, key, q):
    values = [row.get(key) for row in rows if isinstance(row.get(key), (int, float))]
    return float(pd.Series(values).quantile(q)) if values else 0.0


RETRIEVAL_METRIC_KEYS = [f'{name}@{k}' for k in K_VALUES for name in ['recall', 'hit', 'mrr', 'ndcg', 'precision']]
GENERATION_METRIC_KEYS = ['exact_match', 'token_f1', 'rouge_l']
LATENCY_KEYS = ['embedding_latency_sec', 'dense_search_latency_sec', 'bm25_search_latency_sec', 'fusion_latency_sec', 'graph_latency_sec', 'generation_latency_sec', 'total_latency_sec']


def summarize(cases, label):
    if label == 'answerable_only':
        rows = [row for row in cases if row['is_answerable']]
    elif label == 'unanswerable_only':
        rows = [row for row in cases if not row['is_answerable']]
    else:
        rows = list(cases)
    summary = []
    for mode in ['HybridOnly', 'HybridGraph']:
        mode_rows = [row for row in rows if row['mode'] == mode]
        unanswerable_rows = [row for row in mode_rows if not row['is_answerable']]
        item = {
            'summary_set': label,
            'mode': mode,
            'evaluated': len(mode_rows),
            'answerable': sum(row['is_answerable'] for row in mode_rows),
            'unanswerable': len(unanswerable_rows),
            'unanswerable_accuracy': average(unanswerable_rows, 'unanswerable_correct') if unanswerable_rows else None,
            'avg_retrieved_count': average(mode_rows, 'retrieved_count'),
            'avg_graph_added_count': average(mode_rows, 'graph_added_count'),
        }
        for key in RETRIEVAL_METRIC_KEYS + GENERATION_METRIC_KEYS + LATENCY_KEYS:
            item[key] = average(mode_rows, key)
        item['median_total_latency_sec'] = percentile(mode_rows, 'total_latency_sec', 0.50)
        item['p95_total_latency_sec'] = percentile(mode_rows, 'total_latency_sec', 0.95)
        summary.append(item)
    return pd.DataFrame(summary)


summary_all = summarize(all_cases, 'all_cases')
summary_answerable = summarize(all_cases, 'answerable_only')
summary_unanswerable = summarize(all_cases, 'unanswerable_only')
summary_combined = pd.concat([summary_all, summary_answerable, summary_unanswerable], ignore_index=True)

display(summary_combined)


In [ ]:
def write_jsonl(path, rows):
    with path.open('w', encoding='utf-8') as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + '\n')


def csv_safe(df):
    out = df.copy()
    for col in out.columns:
        if out[col].map(lambda x: isinstance(x, (list, dict))).any():
            out[col] = out[col].map(lambda x: json.dumps(x, ensure_ascii=False) if isinstance(x, (list, dict)) else x)
    return out


cases_df = pd.DataFrame(all_cases)
summary_combined = pd.concat([summary_all, summary_answerable, summary_unanswerable], ignore_index=True)

write_jsonl(OUT_DIR / 'e2e_cases.jsonl', all_cases)
csv_safe(cases_df).to_csv(OUT_DIR / 'e2e_cases.csv', index=False, encoding='utf-8-sig')
summary_combined.to_csv(OUT_DIR / 'summary_all_answerability.csv', index=False, encoding='utf-8-sig')
summary_all.to_csv(OUT_DIR / 'summary_all_cases.csv', index=False, encoding='utf-8-sig')
summary_answerable.to_csv(OUT_DIR / 'summary_answerable_only.csv', index=False, encoding='utf-8-sig')
summary_unanswerable.to_csv(OUT_DIR / 'summary_unanswerable_only.csv', index=False, encoding='utf-8-sig')

manifest = {
    'run_name': 'hybrid_graph_generate_eval',
    'modes': ['HybridOnly', 'HybridGraph'],
    'benchmark_path': str(QA_PATH),
    'faiss_dir': str(FAISS_INPUT_DIR),
    'bm25_dir': str(BM25_INPUT_DIR),
    'graph_path': str(GRAPH_PATH),
    'embedding_model': EMBEDDING_MODEL,
    'generation_model': GENERATION_MODEL,
    'base_url': BASE_URL,
    'search_k': SEARCH_K,
    'final_top_n': FINAL_TOP_N,
    'k_values': K_VALUES,
    'rrf_k': RRF_K,
    'bm25_max_shards_in_memory': BM25_MAX_SHARDS_IN_MEMORY,
    'bm25_precompute': BM25_PRECOMPUTE,
    'bm25_precompute_latency_sec': BM25_PRECOMPUTE_LATENCY_SEC,
    'bm25_amortized_latency_sec': BM25_AMORTIZED_LATENCY_SEC,
    'eval_start': run_start_index,
    'eval_end': run_end_index,
    'run_label': RUN_LABEL,
    'retrieval_metrics': RETRIEVAL_METRIC_KEYS,
    'generation_metrics': GENERATION_METRIC_KEYS,
    'latency_metrics': LATENCY_KEYS,
    'graph_seed_n': GRAPH_SEED_N,
    'graph_max_hop': GRAPH_MAX_HOP,
    'graph_context': GRAPH_CONTEXT,
    'filter_profile': FILTER_PROFILE,
    'sample_limit': SAMPLE_LIMIT,
    'qa_total': len(qa_rows),
    'answerable_total': answerable_count,
    'unanswerable_total': len(qa_rows) - answerable_count,
    'total_benchmark_time_sec': total_benchmark_time,
}

with (OUT_DIR / 'manifest.json').open('w', encoding='utf-8') as f:
    json.dump(manifest, f, ensure_ascii=False, indent=2)

print('Saved outputs to:', OUT_DIR)
for path in sorted(OUT_DIR.iterdir()):
    print(' -', path.name)


In [ ]:
ZIP_PATH = Path('/kaggle/working/hybrid_graph_eval_outputs.zip')

if ZIP_PATH.exists():
    ZIP_PATH.unlink()

shutil.make_archive(str(ZIP_PATH.with_suffix('')), 'zip', OUT_DIR)

print('Created:', ZIP_PATH)
print(f'Size: {ZIP_PATH.stat().st_size / 1024**2:.1f} MB')
